In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [4]:
import kagglehub
path = kagglehub.dataset_download("milapgohil/play-tennis-dataset-weather-based-classifier")

100%|██████████| 20.6k/20.6k [00:00<00:00, 16.6MB/s]

Extracting files...


In [6]:
df = pd.read_csv(path + "/play_tennis_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6666 entries, 0 to 6665
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Day          6666 non-null   object
 1   Outlook      6267 non-null   object
 2   Temperature  6333 non-null   object
 3   Humidity     6433 non-null   object
 4   Wind         6300 non-null   object
 5   Play         6666 non-null   object
dtypes: object(6)
memory usage: 312.6+ KB


In [9]:
def impute_categorical_missing(df, categorical_cols):
    for col in categorical_cols:
        df[col].fillna(df[col].mode()[0], inplace=True)
    return df

# List of categorical columns to impute
categorical_columns_to_impute = ['Outlook', 'Temperature', 'Humidity', 'Wind']

# Apply the function
df = impute_categorical_missing(df, categorical_columns_to_impute)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6666 entries, 0 to 6665
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Day          6666 non-null   object
 1   Outlook      6666 non-null   object
 2   Temperature  6666 non-null   object
 3   Humidity     6666 non-null   object
 4   Wind         6666 non-null   object
 5   Play         6666 non-null   object
dtypes: object(6)
memory usage: 312.6+ KB


In [10]:
for column in df.columns:
    print(f"Unique values in '{column}': {df[column].unique()}")

Unique values in 'Day': ['D1' 'D2' 'D3' 'D4' 'D5' 'D6' 'D7' 'D8' 'D9' 'D10' 'D11' 'D12' 'D13'
 'D14' 'D15' 'D16' 'D17' 'D18' 'D19' 'D20' 'D21' 'D22' 'D23' 'D24' 'D25'
 'D26' 'D27' 'D28' 'D29' 'D30' 'D31']
Unique values in 'Outlook': ['Overcast' 'Sunny' 'Rainy']
Unique values in 'Temperature': ['Mild' 'Cool' 'Hot']
Unique values in 'Humidity': ['Normal' 'High']
Unique values in 'Wind': ['Strong' 'Weak']
Unique values in 'Play': ['Yes' 'No']


In [11]:
X = df.drop(columns=['Day','Play'], axis=1)
y = df['Play']

In [29]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [30]:
class NaiveBayesClassifier:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.n_classes = len(self.classes)
        self.n_features = X.shape[1]

        self.priors = np.zeros(self.n_classes)
        self.likelihoods = {}

        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            self.priors[i] = len(X_c) / len(X)

            self.likelihoods[c] = {}
            for feature_idx in range(self.n_features):
                feature_name = X.columns[feature_idx]
                self.likelihoods[c][feature_name] = {}

                # Calculate likelihoods for each unique value in the feature
                unique_feature_values = X[feature_name].unique()
                for value in unique_feature_values:
                    count_value_in_class = (X_c[feature_name] == value).sum()
                    # Add Laplace smoothing to prevent zero probabilities
                    self.likelihoods[c][feature_name][value] = (count_value_in_class + 1) / (len(X_c) + len(unique_feature_values))

    def predict(self, X):
        predictions = [self._predict_single_sample(x) for _, x in X.iterrows()]
        return np.array(predictions)

    def _predict_single_sample(self, x):
        posteriors = []

        for i, c in enumerate(self.classes):
            prior = np.log(self.priors[i])
            likelihood = 0
            for feature_idx in range(self.n_features):
                feature_name = x.index[feature_idx]
                feature_value = x[feature_name]

                # Get the likelihood from the stored dictionary, handle unseen values
                if feature_value in self.likelihoods[c][feature_name]:
                    likelihood += np.log(self.likelihoods[c][feature_name][feature_value])
                else:
                    # Handle unseen feature values during prediction (use smoothing term)
                    unique_feature_values = self.likelihoods[c][feature_name].keys()
                    likelihood += np.log(1 / (len(self.likelihoods[c]) + len(unique_feature_values)))

            posterior = prior + likelihood
            posteriors.append(posterior)

        return self.classes[np.argmax(posteriors)]

In [31]:
# Instantiate and train the Naive Bayes classifier
nb_classifier = NaiveBayesClassifier()
nb_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = nb_classifier.predict(X_test)

In [32]:
from sklearn.metrics import accuracy_score

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 80.55%
